# TrkA 결합여 설계 · RFdiffusion3 → MPNN → 교정된 AF2 필터 · v0.1.0

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/notebooks/03_design_trka_T4.ipynb?v=0.1.0)

T4에서 전체 실행하면 **새 결합여를 직접 설계하고 채점합니다.** 지금까지는 남의 설계를 재현하고 채점했고, 이 노트북이 설계 단계입니다.

| 단계 | 도구 | 산출 |
| --- | --- | --- |
| 1 | RFdiffusion3 | 백본 20개 |
| 2 | LigandMPNN | 백본당 서열 2개 = 설계 40개 |
| 3 | AF2 initial guess | 설계 40개 + 양성 대조군 1개 채점 |

환경이 둘입니다. RFdiffusion3·MPNN은 Python 3.12의 foundry 스택을, AF2는 Python 3.11의 JAX·PyRosetta를 씁니다. 노트북이 두 환경을 각각 띄워 순서대로 돌립니다. 같은 세션이면 설치와 가중치를 재사용합니다.

## 표적과 hotspot

공개 결합여가 실제로 붙는 자리를 그대로 겨냥합니다. TrkA D5 도메인을 설계 좌표계에 두고, 사이드체인은 결정 구조(1WWW 체인 X)에서 이식한 것을 씁니다.

| hotspot | 잔기 | 공개 결합여와의 최단거리 |
| --- | --- | --- |
| B108 | MET | 3.21 Å |
| B115 | PHE | 3.80 Å |
| B145 | LEU | 3.15 Å |

계면 26잔기 중 가장 깊이 묻힌 소수성 잔기 셋입니다. 임의의 표면이 아니라 결합이 확인된 에피토프입니다.

## 양성 대조군

채점 배치에 공개 결합여 **TrkA_88을 함께 넣습니다**. 실험으로 결합이 확인됐고 이 저장소가 1.496 Å / `pae_interaction` 5.86으로 재현한 설계입니다. 결과표에 항상 기준점 한 줄이 있게 됩니다.

## 통과 기준과 해석

논문 기준 그대로 `pae_interaction < 10`, `plddt_binder > 80`, `binder_aligned_rmsd < 1 Å`입니다.

**통과 개수만으로 성공을 판단하지 않습니다.** 이 저장소의 [교정 결과](https://github.com/HisameOgasahara/tmp_protein/blob/main/docs/02-filter-calibration.md)에서 실험상 결합하지 않은 설계 14개가 전부 이 기준을 통과했습니다. 통과는 합격이 아니라 기본 확률을 얻는 것이고, 이 표적에서 그 값은 6/95 ≈ 6.3%입니다.

이번 설계는 **걸러지지 않은 날것**이라는 점이 교정 집합과 다릅니다. 통과선 아래로 떨어지는 설계가 나오면, 그게 이 필터가 앞단에서 실제로 일한다는 직접 증거가 됩니다.

[설계 사양](https://github.com/HisameOgasahara/tmp_protein/blob/main/inputs/trka/design_spec.json) · [코드](https://github.com/HisameOgasahara/tmp_protein/blob/main/src/rfd3_design/trka_workflow.py)


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="trka_design_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")


In [ ]:
import json
import pandas as pd
from IPython.display import display

log = checkout.parent / "trka_design.log"
result_zip = None
with log.open("w", encoding="utf-8") as handle:
    with subprocess.Popen(
        [sys.executable, "-u", str(checkout / "src/rfd3_design/trka_workflow.py"), revision],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
    ) as process:
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            print(line, end="", flush=True)
            if line.strip().endswith(".zip"):
                result_zip = Path(line.strip())
        if process.wait():
            raise RuntimeError(f"Design run failed. Full log: {log}")
if result_zip is None or not result_zip.is_file():
    raise FileNotFoundError(f"Missing result ZIP. Full log: {log}")

folder = result_zip.with_suffix("")
table = folder / "designs.csv"
if not table.is_file():
    print("설계 또는 채점 단계 오류 기록이 ZIP에 저장됐습니다.")
else:
    frame = pd.read_csv(table)
    columns = ["name", "control", "passes_filter", "pae_interaction", "plddt_binder",
               "binder_aligned_rmsd", "target_aligned_rmsd", "rmsd_vs_design", "length"]
    print()
    print("=== 설계별 점수 (결합면 PAE 순, control = 공개 결합여) ===")
    display(frame.reindex(columns=columns).sort_values("pae_interaction").round(3))
    print()
    print("=== 교정 집합과 비교 ===")
    reference = checkout / "results/2026-11-18-trka88-calibration/calibration.csv"
    if reference.is_file():
        known = pd.read_csv(reference)
        rows = [{"집단": "실험 결합 (6)", "출처": "공개",
                 "PAE 최소": known[known.binder_experimental]["pae_interaction"].min(),
                 "PAE 중앙": known[known.binder_experimental]["pae_interaction"].median(),
                 "PAE 최대": known[known.binder_experimental]["pae_interaction"].max()},
                {"집단": "실험 비결합 (14)", "출처": "공개",
                 "PAE 최소": known[~known.binder_experimental]["pae_interaction"].min(),
                 "PAE 중앙": known[~known.binder_experimental]["pae_interaction"].median(),
                 "PAE 최대": known[~known.binder_experimental]["pae_interaction"].max()}]
        new = frame[(~frame.control) & (frame.status == "success")]
        if len(new):
            rows.append({"집단": f"이번 설계 ({len(new)})", "출처": "신규",
                         "PAE 최소": new["pae_interaction"].min(),
                         "PAE 중앙": new["pae_interaction"].median(),
                         "PAE 최대": new["pae_interaction"].max()})
        display(pd.DataFrame(rows).round(3))
    print()
    summary = folder / "designs_summary.json"
    if summary.is_file():
        print(json.dumps(json.loads(summary.read_text()), indent=2, ensure_ascii=False))
print(result_zip)


In [ ]:
from google.colab import files
files.download(str(result_zip))
